In [29]:
import torch
import torch.nn as nn
from torch.nn.utils import prune
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.utils.data as data
from torch.utils.data import Subset
import torch_pruning as tp
from ptflops import get_model_complexity_info
from ultralytics import YOLO
import os

from yolo_test import test_yolo_model

## Helper Functions

In [30]:
# Calculate FLOPs and parameters (adjusted for 1-channel input)
def print_model_flops(model, input_res=(1, 3, 32, 32), name="Model"):
    macs, params = get_model_complexity_info(model, input_res, as_strings=True,
                                             print_per_layer_stat=False, verbose=False)
    print(f"{name} – FLOPs: {macs}, Parameters: {params}")


In [ ]:
from ultralytics.utils.torch_utils import prune
def prune_yolo_model(model: YOLO, prune_rate: float, save_path: str):
    """
    对 YOLO 模型进行剪枝并保存。

    Args:
        model (YOLO): 预加载的 YOLO 模型对象 (e.g., YOLO('yolov11m.pt')).
        prune_amount (float): 剪枝的比例，介于 0.0 到 1.0 之间 (e.g., 0.5 表示剪枝 50%).
        save_path (str): 剪枝后模型的保存路径 (e.g., 'yolov11m_pruned.pt').
    """
    prune(model, prune_rate)
    model.save(save_path)

    print("--- 剪枝过程完成 ---")


ImportError: cannot import name 'prune' from 'ultralytics.utils.torch_utils' (/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/ultralytics/utils/torch_utils.py)

## Training and Evaluation

In [ ]:
# Prepare model
model = YOLO("runs/detect/train3/weights/best.pt")
data_yml = "train.yaml"

device = "cuda:0"
# FLOPs & Accuracy before pruning
print_model_flops(model, name="Before Pruning")

# Example input for pruning
example_inputs = torch.randn(1, 1, 32, 32).to(device)

# Pruning setup
importance = tp.importance.MagnitudeImportance(p=1)  # L1 norm

# AGP setup: start at 10%, end at 50% sparsity over 3 steps
num_pruning_steps = 3
epochs_per_step = 5
initial_ratio = 0.1
final_ratio = 0.5
ratios = torch.linspace(initial_ratio, final_ratio, steps=num_pruning_steps)

for step, ratio in enumerate(ratios):
    print(f"=== Pruning Step {step+1}/{num_pruning_steps} (Ratio: {ratio:.2f}) ===")

    pruned_save_path = f"prunes/save{step}.pt"
    prune_yolo_model(model, float(ratio), pruned_save_path)

    pruned_model = YOLO(pruned_save_path)

    # train_epochs(model, device, trainloader, epochs=epochs_per_step)
    train_results = pruned_model.train(
        data=data_yml,  # Path to dataset YAML (must be segmentation-compatible)
        epochs=epochs_per_step,  # Number of training epochs
        imgsz=32,  # Image size
        device=0,  # GPUs to use (or "cpu" for CPU training)
        batch=320,  # Adjust batch size based on GPU memory
        workers=4,  # Number of dataloader workers
        optimizer="AdamW",  # AdamW optimizer (optional, can use "SGD")
        lr0=0.01,  # Initial learning rate
        lrf=0.01,
        patience=50,  # Early stopping patience
        seed=42,  # Random seed for reproducibility
        verbose=True, # Display training progress
        multi_scale=False,
        pretrained = True,
        single_cls = False,
        cos_lr=True,
        box = 15,
    )
    print_model_flops(pruned_model.model, name=f"After Pruning Step {step+1}")
    # evaluate_model(model, testloader, device, name=f"After Pruning Step {step+1}")
    test_yolo_model(pruned_save_path, data_yml, 32, 320, device=device)


print("Cov Pruning (AGP) + Fine-Tuning completed!")



Flops estimation was not finished successfully because of the following exception:
<class 'ValueError'> : torch.Tensor inputs should be BCHW i.e. shape(1, 3, 640, 640) divisible by stride 32. Input shape(1, 1, 3, 32, 32) is incompatible.
Before Pruning – FLOPs: None, Parameters: None
=== Pruning Step 1/3 (Ratio: 0.10) ===
--- 开始对 YOLO 模型进行剪枝 (剪枝比例: 10.00%) ---
成功对 113 个 Conv2d 层应用了剪枝。
正在移除剪枝的重新参数化，使其永久化...
成功移除了 113 个层的重新参数化。
剪枝后的模型已成功保存到: prunes/save0.pt
--- 剪枝过程完成 ---


Traceback (most recent call last):
  File "/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/ptflops/pytorch_engine.py", line 64, in get_flops_pytorch
    _ = flops_model(batch)
  File "/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/ultralytics/engine/model.py", line 182, in __call__
    return self.predict(source, stream, **kwargs)
  File "/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/ultralytics/engine/model.py", line 552, in predict
    return self.predictor.predict_cli(source=source) if is_cli else self.predictor(source=source, stream=stream)
  File "/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/ultralytics/engine/predictor.py", line 218, in __call__
    return list(self.stream_inference(source, model, *args, **kwargs))  # merge list of Result into one
  File "/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/torch/utils/_contextlib.py", line 36, in generator_context
    respon

New https://pypi.org/project/ultralytics/8.3.162 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.134 🚀 Python-3.8.0 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060, 7899MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=320, bgr=0.0, box=15, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=train.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=32, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=prunes/save0.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train5, nbs=64, nms=False, opset=None, optimize=False, opt

train: Scanning /home/supercomputing/studys/traffic_sign/dataset/yolo_dataset/train/labels.cache... 33328 images, 0 backgrounds, 0 corrupt: 100%|██████████| 33328/33328 [00:00<?, ?it/s]


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 402.4±245.1 MB/s, size: 5.4 KB)


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        2/5      1.93G       1.28      0.797     0.8972        121         32: 100%|██████████| 105/105 [00:09<00:00, 11.48it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 10/10 [00:05<00:00,  1.90it/s]


                   all       5881       5881   1.41e-05     0.0059   8.84e-06   2.71e-06

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        3/5      1.95G       1.23     0.7676     0.8943        825         32:  94%|█████████▍| 99/105 [00:09<00:00, 10.72it/s]


KeyboardInterrupt: 